# PROJECT 6
by Szymon Waliczek
 - Ballistic **Andreev** transport in **2DEG** side **Josephson junction** - semi/super-conducting hybrid **InAs-SC**
 - System is a wire with **2 semiconducting leads(up/down)** and **1 superconducting lead/block** on the right edge.
 - We analyze the relation of dispersion using wrapped system with Y translational symmetry
 - **With Peierls phase**
 - **No spin**
 - With **type s** electron pairing

In [ ]:
import ipyparallel as ipp
#-------------------------------------------------------------------------------------------------------------------
cluster = ipp.Client(profile="kwant_parallel")
#-------------------------------------------------------------------------------------------------------------------
#from _tera import Tera_Client
#cluster = TeraClient(username="SWALICZEK", profile_name="slurm_cpu")
#-------------------------------------------------------------------------------------------------------------------
lview = cluster.load_balanced_view()
len(cluster[:])

In [ ]:
%%px --local

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
import pickle
import adaptive
adaptive.notebook_extension()
import ipywidgets as widgets
from time import perf_counter
from IPython.display import display
from matplotlib import pyplot as plt
from ipywidgets import interactive, HBox, VBox, fixed
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
#-------------------------------------------------------------------------------------------------------------------
class Timer():
    def __init__(self, name):
        self.name = name
        self.engine_id = os.environ.get('IPY_ENGINE_ID', '0')
    
    def __enter__(self):
        self.start = perf_counter()
        return self

    def __exit__(self, *args):
        self.end = perf_counter()
        if self.engine_id == '0':
            t = self.end - self.start
            print(f"Time - {self.name}: {t/60:.3f} min")

In [ ]:
%%px --local

import kwant
import tinyarray
import numpy as np
from functools import lru_cache
from scipy.sparse.linalg import eigsh
#-------------------------------------------------------------------------------------------------------------------
tau_0 = tinyarray.array([[1, 0], [0, 1]])
tau_x = tinyarray.array([[0, 1], [1, 0]])
tau_y = tinyarray.array([[0, -1j], [1j, 0]])
tau_z = tinyarray.array([[1, 0], [0, -1]])
#-------------------------------------------------------------------------------------------------------------------
# Physical constants
from scipy.constants import physical_constants
eV = physical_constants['electron volt'][0]
m_el = physical_constants['electron mass'][0]
h_bar = physical_constants['Planck constant over 2 pi'][0]
phi_0 = physical_constants['elementary charge over h-bar'][0]
#-------------------------------------------------------------------------------------------------------------------
a = 10
m_eff = 0.023 * m_el
t = (h_bar**2 / (2 * m_eff * (a*1e-9)**2)) / eV # hopping in eV
PI = np.pi

In [ ]:
%%px --local

def onsite(site, mu, B, delta, x0):
    return (4*t - mu) * tau_z
#-------------------------------------------------------------------------------------------------------------------
def onsite_sc(site, mu, B, delta, x0):
    return (4*t - mu)*tau_z + delta*tau_x
#-------------------------------------------------------------------------------------------------------------------
def hop(site1, site2, mu, B, delta, x0):
    x1, y1 = site1.pos
    x2, y2 = site2.pos
    if x1 < 0:
        phi = phi_0 * B * ((x1 + x2)/2 + x0) * (y1 - y2)*1e-18
        p_phase = tinyarray.array([[np.exp(-1j * phi), 0], 
                              [0, np.exp(1j * phi)]])
        return -t * tau_z * p_phase
    return -t * tau_z * 1.0 # A=0
#-------------------------------------------------------------------------------------------------------------------    
def make_systems(a, W, W_sc_block, L, t):
    # SC hybrid
    lat = kwant.lattice.square(a, norbs=2)
    sys_hybrid = kwant.Builder()
    sys_hybrid[lat.shape(lambda pos: -W <= pos[0] < 0 and -L/2 <= pos[1] <= L/2, (-a, 0))] = onsite # Site N
    sys_hybrid[lat.shape(lambda pos: pos[0] == 0 and -L/2 <= pos[1] <= L/2, (0, 0))] = onsite_sc # Site SC
    sys_hybrid[lat.neighbors()] = hop
    lead_n = kwant.Builder(kwant.TranslationalSymmetry((0, -a)), conservation_law=-tau_z, particle_hole=tau_y) # Lead N
    lead_n[lat.shape(lambda pos: -W <= pos[0] < 0, (-a, 0))] = onsite
    lead_n[lat.neighbors()] = hop
    sys_hybrid.attach_lead(lead_n);
    sys_hybrid.attach_lead(lead_n.reversed());
    lead_sc = kwant.Builder(kwant.TranslationalSymmetry((a, 0))) # Lead SC
    lead_sc[lat.shape(lambda pos: -L/2 <= pos[1] <= L/2, (0, 0))] = onsite_sc
    lead_sc[lat.neighbors()] = hop
    sys_hybrid.attach_lead(lead_sc)
    #-------------------------------------------------------------------------------------------------------------------
    # SC block
    sys_block = kwant.Builder()
    sys_block[lat.shape(lambda pos: -W <= pos[0] < 0 and -L/2 <= pos[1] <= L/2, (-a, 0))] = onsite # Site N
    if(W_sc_block != 0): sys_block[lat.shape(lambda pos: 0 <= pos[0] <= W_sc_block and -L/2 <= pos[1] <= L/2, (0, 0))] = onsite_sc # Site SC
    sys_block[lat.neighbors()] = hop
    sys_block.attach_lead(lead_n)
    sys_block.attach_lead(lead_n.reversed())
    # Normal
    if(W_sc_block == 0): return sys_block.finalized()
    return sys_hybrid.finalized(), sys_block.finalized()
#-------------------------------------------------------------------------------------------------------------------
def make_wrap_system(a, W, W_sc_wrap, L, t):
    # Wraparound
    lat = kwant.lattice.square(a, norbs=2)
    sys_wrap = kwant.Builder(kwant.TranslationalSymmetry((0, a)))
    if(W != 0): sys_wrap[lat.shape(lambda pos: -W <= pos[0] < 0, (-a, 0))] = onsite
    if(W_sc_wrap != 0): sys_wrap[lat.shape(lambda pos: 0 <= pos[0] <= W_sc_wrap, (0, 0))] = onsite_sc
    sys_wrap[lat.neighbors()] = hop
    return kwant.wraparound.wraparound(sys_wrap, coordinate_names='y').finalized()
#-------------------------------------------------------------------------------------------------------------------
@lru_cache(maxsize=1)
def initialize_wrap(_a, _W, _W_sc_wrap, _L, _t):
    ini_wrap = make_wrap_system(a=_a, W=_W, W_sc_wrap=_W_sc_wrap, L=_L, t=_t)
    return ini_wrap
#-------------------------------------------------------------------------------------------------------------------
def compute_eigen(k, sys_wrap, modes, mu, B, delta, x0):
    params = dict(mu=mu, B=B, delta=delta, x0=x0, k_y=k)
    H = sys_wrap.hamiltonian_submatrix(params=params, sparse=True)
    evals, evecs = eigsh(H, k=modes, sigma=0, which='LM')
    idx = evals.real.argsort()
    return evals[idx].real, evecs[:, idx]
#-------------------------------------------------------------------------------------------------------------------
def eigen(a, W, WSC, L, t, modes, xlim, nk, p):
    k_range = np.linspace(-xlim, xlim, nk)
    results = lview.map_sync(lambda k: compute_eigen(k, initialize_wrap(a, W, WSC, L, t),
                                                         modes, p['mu'], p['B'], p['delta'], p['x0']), k_range)
    E = np.array([x[0] for x in results])
    V = np.array([x[1] for x in results])
    return E, k_range, V
#-------------------------------------------------------------------------------------------------------------------
def compute_Gj(sys, E, p, j):
    smatrix = kwant.smatrix(sys, E, params=p)
    if j == 0:
        n0 = smatrix.submatrix((0, 0), (0, 0)).shape[0]
        ree = smatrix.transmission((0, 0), (0, 0))
        rhe = smatrix.transmission((0, 1), (0, 0))
        return n0 - ree + rhe
    tee = smatrix.transmission((j, 0), (0, 0))
    the = smatrix.transmission((j, 1), (0, 0))
    print(f"{tee-the}\n")
    return tee - the
#-------------------------------------------------------------------------------------------------------------------
def compute_modes_lead(lead, _mu, _B, _delta, _x0, _E):
    p = dict(mu=_mu, B=_B, delta=_delta, x0=_x0)
    prop_modes, _ = lead.modes(energy=_E, params=p)
    return len(prop_modes.momenta)
#-------------------------------------------------------------------------------------------------------------------
@lru_cache(maxsize=1)
def initialize_lead(a, W, Wscblock, L, t):
    lat = kwant.lattice.square(a, norbs=2) 
    lead_hybrid = kwant.Builder(kwant.TranslationalSymmetry((0, -a)), particle_hole=tau_y)
    lead_hybrid[lat.shape(lambda pos: -W <= pos[0] < 0, (-a, 0))] = onsite
    if Wscblock > 0: lead_hybrid[lat.shape(lambda pos: 0 <= pos[0] < Wscblock, (0, 0))] = onsite_sc
    lead_hybrid[lat.neighbors()] = hop
    return lead_hybrid.finalized()
#-------------------------------------------------------------------------------------------------------------------
def get_dk_lead(_a, _W, _WSC, _L, _t, _delta, _x0, _mu, _B):
    p = dict(mu=_mu, B=_B, delta=_delta, x0=_x0)
    prop_modes, _ = initialize_lead(_a, _W, _WSC, _L, _t).modes(energy=0, params=p)
    momentum = prop_modes.momenta
    k_vals = np.sort(momentum[momentum > 0])
    if len(k_vals) == 0: return 0.0
    k_p = k_vals[0]
    return 2 * k_p

In [ ]:
def plot_sys(sys, x, y, name, p):
    with Timer(name):
        COLOR = lambda site: np.real(sys.hamiltonian(site, site, params = p)[0,1])
        kwant.plot(sys, fig_size=(x, y), show=False, site_color = COLOR)
        plt.title(f"{name} a = {a}nm")
        plt.xlabel("x [nm]")
        plt.ylabel("y [nm]")
        plt.show()
#-------------------------------------------------------------------------------------------------------------------
def plot_bands_wrap(name, E, k_range, dk, W, L, s, xlim, ylim, p):
    modes = E.shape[1]
    plt.figure(figsize=(s,s))
    for i in range(modes):
        plt.plot(k_range, E[:, i]*1e3, 'k.', markersize=1)
    info = (f"{name} \nW = {W} nm \nL = {L} nm \n$\mu = {p['mu']*1e3:.0f}$ meV \n$B = {p['B']:.1f}$ T\
    \n$\Delta = {p['delta']*1e3}$ meV \n$\delta_k = {dk:.3f}$ 1/a")
    plt.text(1.1, 0.5, info, transform=plt.gca().transAxes, fontsize=12, verticalalignment='center')
    plt.hlines(y=0, xmin=-dk/2, xmax=dk/2, colors='red', label='$\delta_k$', lw=1)
    plt.axhline(p['delta']*1e3, color='blue', label='$\Delta$', lw=0.2)
    plt.axhline(p['mu']*1e3, color='seagreen', label='$\mu$', lw=0.3)
    plt.axhline(-p['delta']*1e3, color='blue', lw=0.2)
    plt.ylabel("E [meV]"); plt.yticks(np.arange(-ylim, ylim+ylim/10, ylim/5)); plt.ylim(-ylim, ylim)
    plt.xlabel("$k_y$ [1/a]"); plt.xlim(-xlim, xlim); plt.xticks(np.arange(-xlim, xlim+xlim/10, xlim/2))
    plt.legend(loc='upper right', fontsize=2*s); plt.grid(alpha=0.3); plt.show
#-------------------------------------------------------------------------------------------------------------------
def interactive_E_k(_name, _s, _a, _W, _WSC, _L, _t, _modes, _xlim, _ylim, _nk, _mu, _B, _delta, _x0):
    _p = dict(mu=_mu, B=_B, delta=_delta, x0=_x0)
    dk_lead = get_dk_lead(_a=_a, _W=_W, _WSC=_WSC, _L=_L, _t=_t, _mu=_mu, _B=_B, _delta=_delta, _x0=_x0)
    Evals, k, _ = eigen(a=_a, W=_W, WSC=_WSC, L=_L, t=_t, modes=_modes, xlim=_xlim, nk=_nk, p=_p)
    plot_bands_wrap(name=_name, E=Evals, k_range=k, dk=dk_lead, W=_W, L=_L, s=_s, xlim=_xlim, ylim=_ylim, p=_p)
#-------------------------------------------------------------------------------------------------------------------
def compute_G_mu_B(_sys, _E, _delta, _x0, _mu, _B, _j, A):
    def map_G(tuple):
        mu_val, B_val = tuple
        p = dict(mu=mu_val, B=-B_val, delta=_delta, x0=_x0)
        return compute_Gj(sys=_sys, E=_E, p=p, j=_j)
    learner = adaptive.Learner2D(map_G, bounds=[_mu, _B])
    runner = adaptive.Runner(learner, executor = cluster, goal=lambda l: l.loss() < A)
    runner.live_info()
    return learner, runner, int(A*1000)
#-------------------------------------------------------------------------------------------------------------------
def to_file(l, r, a, filename):
    with open(f"{filename}_00{a}_data.pkl", "wb") as f: pickle.dump(l.data, f)
    t = r.elapsed_time()
    P = len(l.data)
    del l; del r
    return f"{filename}_00{a}_data.pkl", t, P
#-------------------------------------------------------------------------------------------------------------------
def plot_G_mu_B(E, title, name, L, s, delta, x0, _mu, _B, t, P, N, a, v_map, Nv, filename, xs_mu, xs_B):
    with Timer('plot_G_mu_B'):
        with open(filename, "rb") as f: data = pickle.load(f)
        tmp_learner = adaptive.Learner2D(lambda x: x, bounds=[_mu, _B])
        tmp_learner.data = data
        del data
        x, y, z = tmp_learner.interpolated_on_grid(n=N) # !!! z = [len(y), len(z)] !!!
        del tmp_learner
        v_abs = max(abs(z.min()), abs(z.max()))
        plt.figure(figsize=(1.6*s, s))
        im = plt.pcolormesh(x*1e3, y, z.T, cmap='seismic', vmin=-v_abs, vmax=v_abs)
        info = (f"{title} \n$L = {L}$ nm \n$E = {E*1e3:.2f}$ meV\n$\Delta = {int(delta*1e3)}$ meV \
                \n$t = {int(t/60)}$ min \n$p = {int(P/1000)}$k | ${int(N/1000)}$k \n$Nv = {Nv}$")
        plt.text(1.2, 0.5, info, transform=plt.gca().transAxes, fontsize=3*s, verticalalignment='center')
        plt.colorbar(im)
        plt.xlabel('$\mu$ [meV]'); plt.ylabel('$-B$ [T]')
        plt.title(f'G ( $\mu$, -B ) [ $e^2/h$ ]')
        plt.tight_layout()
#-------------------------------------------------------------------------------------------------------------------
        mu_vec = np.linspace(_mu[0], _mu[1], v_map.shape[0]) * 1e3
        B_vec = np.linspace(_B[0], _B[1], v_map.shape[1])
        levels = [1, 5, 9, 13]
        cs = plt.contour(mu_vec, B_vec, v_map.T, levels=levels, colors='black', linewidths=0.6)
        plt.clabel(cs, inline=True, fontsize=2*s, fmt={1:'2', 5:'4', 9:'6', 13:'8'})
#-------------------------------------------------------------------------------------------------------------------
        if(xs_B != None): plt.hlines(y=-xs_B, xmin=_mu[0], xmax=_mu[1], colors='saddlebrown', label='$xs_B$', lw=0.8)
        if(xs_mu != None): plt.vlines(ymin=_B[0], ymax=_B[1], x=xs_mu, colors='seagreen', label='$xs_B$', lw=0.8)
        if((xs_B != None) and (xs_mu != None)):
            plt.hlines(y=-xs_B, xmin=xs_mu[0], xmax=xs_mu[1], colors='saddlebrown', label='$xs_B$', lw=0.8)
#-------------------------------------------------------------------------------------------------------------------
        plt.savefig(f"{name}_Nv{Nv}_P{P}_N{N}_00{a}_plot.png", dpi=150, bbox_inches='tight')
        plt.close()
#-------------------------------------------------------------------------------------------------------------------
def get_modes_lead(_a, _W, _Wscblock, _L, _t, delta, x0, mu, B, N, E):
    with Timer('modes_lead'):
        mu_vec = np.linspace(mu[0], mu[1], N)
        B_vec = np.linspace(B[0], B[1], N)
        points = [(m, b) for m in mu_vec for b in B_vec]
        lead = initialize_lead(_a, _W, _Wscblock, _L, _t)
        results = lview.map_sync(lambda p: compute_modes_lead(lead, delta, x0, p[0], p[1], E), points)
        map_v = np.array(results).reshape(N, N)
    return map_v, N
#-------------------------------------------------------------------------------------------------------------------
def get_modes_lead_sys(sys, delta, x0, mu, B, N, E):
    with Timer('modes_lead'):
        lead = sys.leads[0]
        mu_vec = np.linspace(mu[0], mu[1], N)
        B_vec = np.linspace(B[0], B[1], N)
        points = [(m, b) for m in mu_vec for b in B_vec]
        results = lview.map_sync(lambda p: compute_modes_lead(lead, delta, x0, p[0], p[1], E), points)
        map_v = np.array(results).reshape(N, N)
    return map_v, N
#-------------------------------------------------------------------------------------------------------------------
def analytical_xs_G(_sys, _a, _W, _WSC, _L, _t, _alpha, _beta, p):
    with Timer('Analytical'):
        mu_vec = np.linspace(p['mu0'], p['mu1'], p['N'])
        def analytical(a, L, alpha, beta, dk): 
            return 1.0 - 8.0*(alpha*beta)**2 * np.sin(dk*L/a/2.0)**2
        results = lview.map_sync(lambda mu: get_dk_lead(_a=_a, _W=_W, _WSC=_WSC, _L=_L, _t=_t, _mu=mu, _B=p['B'], _delta=p['delta'], _x0=p['x0']), mu_vec)
        dk_vec = np.array(results)
        g = lview.map_sync(lambda _dk: analytical(a=_a, L=_L, alpha=_alpha, beta=_beta, dk=_dk), dk_vec)
        return np.array(g)
#-------------------------------------------------------------------------------------------------------------------
def numerical_xs_G(_sys, p1):
    with Timer('Numerical'):
        mu_vec = np.linspace(p1['mu0'], p1['mu1'], p1['N'])
        p_base = [dict(mu=m, B=p1['B'], delta=p1['delta'], x0=p1['x0']) for m in mu_vec]
        g = lview.map_sync(lambda _p: compute_Gj(sys=_sys, E=p1['E'], p=_p, j=1), p_base)
        return np.array(g)
#-------------------------------------------------------------------------------------------------------------------
def plot_xs_G(Ga, Gn, name, L, s, ylim, p):
    plt.figure(figsize=(s, 4*s/5))
    mu_vec = np.linspace(p['mu0'], p['mu1'], p['N'])
    plt.plot(mu_vec*1e3, Ga, 'blue', label=f'Analytical P_e-P_h')
    plt.plot(mu_vec*1e3, Gn, 'red', label=f'Numerical G')
    info = (f"{name} \n$L = {L1200}$ nm \n$E = {p['E']*1e3:.0f}$ meV\n$\Delta = {p['delta']*1e3:.0f}$ meV \n$B={p['B']:.1}$ T")
    plt.text(1.1, 0.5, info, transform=plt.gca().transAxes, fontsize=12, verticalalignment='center')
    plt.ylabel('G / P [$e^2/h$]'); plt.yticks(np.arange(-ylim, ylim+ylim/10, 0.2)); plt.ylim(-ylim, ylim)
    plt.xlabel("$\mu$ [meV]"); plt.xticks(np.arange(p['mu0']*1e3, p['mu1']*1e3+p['mu1']*1e3/10, 1))
    plt.legend(loc='upper right', fontsize='xx-small');    
    plt.title(f'G ( $\mu$ ) [ $e^2/h$ ]')
    plt.grid(True); plt.show()

In [ ]:
p_test = dict(mu = 0, B = 0, delta = 0.001, x0 = 0, k_y=0)

In [ ]:
w_200 = make_wrap_system(a, 0, 200, 0, t)
plot_sys(w_200, 8, 1, 'W_200', p_test)

In [ ]:
Dispersion_relation = interactive(
    interactive_E_k,
    _name=fixed('s-wave'), _s=fixed(4),
    _a=fixed(a), _W=fixed(200), _WSC=fixed(200), _L=fixed(0), _t=fixed(t), 
    _modes=fixed(10), _xlim=fixed(PI/3), _ylim=fixed(4), _nk=fixed(301),
    _mu=widgets.FloatSlider(min=0, max=0.01, step=0.001, value=0.007, description='mu:', readout_format='.3f'),
    _B=widgets.FloatSlider(min=-3, max=0, step=0.1, value=-0.8, description='B:', readout_format='.1f'),
    _delta=widgets.FloatSlider(min=0, max=0.01, step=0.001, value=0.001, description='delta:', readout_format='.3f'),
    _x0=widgets.FloatSlider(min=0, max=10, step=0.1, value=0, description='x0:', readout_format='.1f')
);
controls = VBox(Dispersion_relation.children[:-1])
output = Dispersion_relation.children[-1]
display(HBox([output, controls]))

In [ ]:
%%px --local
# Geometry
L400 = 400
L800 = 800
L1000 = 1000
L1200 = 1200
W1000 = 1000
W2000 = 2000
WSC1000 = 1000
freedom_deg = 2; # e + holes

In [ ]:
h_l_1200, b_l_1200 = make_systems(a, W1000, WSC1000, L1200, t)
w_1000 = make_wrap_system(a, W1000, WSC1000, L1200, t)
plot_sys(h_l_1200, 5, 6, 'H_L_1200', p_test)
plot_sys(w_1000, 10, 1, 'W1000', p_test)

In [ ]:
Gmap_l_1200, R_l_1200, A_l_1200 = compute_G_mu_B(_sys=h_l_1200, _E=0.0, _delta=0.001, _x0=0, _mu=(0, 0.01), _B=(0, 1.5), _j=1, A=0.0005)

In [ ]:
f_l_1200, time_l_1200, points_l_1200 = to_file(l=Gmap_l_1200, r=R_l_1200, a=A_l_1200, filename='6_data/s_h_l_1200')

In [ ]:
Vmap_l_1200, nv_l_1200 = get_modes_lead_sys(h_l_1200, delta=0.001, x0=0.0, mu=(0, 0.01), B=(0, 1.5), N=200, E=0)

In [ ]:
# Vmap_l_1200_lead, nv_l_1200_lead = get_modes_lead(_a=a, _W=W1000, _Wscblock=W1000, _L=L1200, _t=t, delta=0.001, x0=0.0, mu=(0, 0.01), B=(0, 1.5), N=100, E=0)

In [ ]:
plot_G_mu_B(E=0.0, title='s-wave', name='6_plots//sh_l_1200', L=L1200, s=5, delta=0.001, x0=0, _mu=(0, 0.01), _B=(0, 1.5), 
            t=time_1200, P=points_1200, N=5000, a=A_1200, v_map=Vmap_1200, Nv=nv_1200, filename=f_1200, xs_mu=None, xs_B=None)

In [ ]:
h_w_2000, b_w_2000 = make_systems(a, W2000, WSC1000, L1000, t)
w_2000 = make_wrap_system(a, W2000, WSC1000, L1000, t)
plot_sys(h_w_2000, 5, 6, 'H_w_2000', p_test)
plot_sys(w_2000, 10, 1, 'W_2000', p_test)

In [ ]:
Gmap_lw2000, R_w_2000, A_w_2000 = compute_G_mu_B(_sys=h_w_2000, _E=0.0, _delta=0.001, _x0=0, _mu=(0, 0.01), _B=(0, 1.5), _j=1, A=0.0005)

In [ ]:
f_w_2000, time_w_2000, points_w_2000 = to_file(l=Gmap_lw2000, r=R_w_2000, a=A_w_2000, filename='6_data/sh_w_2000')

In [ ]:
Vmap_w_2000, nv_w_2000 = get_modes_lead_sys(h_w_2000, delta=0.001, x0=0.0, mu=(0, 0.01), B=(0, 1.5), N=200, E=0)

In [ ]:
plot_G_mu_B(E=0.0, title='s-wave', name='6_plots//sv2h_w_2000', L=L1000, s=5, delta=0.001, x0=0, _mu=(0, 0.01), _B=(0, 1.5), 
            t=time_w_2000, P=points_w_2000, N=5000, a=A_w_2000, v_map=Vmap_w_2000, Nv=nv_w_2000, filename=f_w_2000, xs_mu=(6, 9), xs_B=-0.8)

# Cross sections  s-wave
# For $\nu = 2$ $(B = -0.8 T)$ $(\mu \approx [3; 6] meV)$

In [ ]:
plot_G_mu_B(E=0.0, title='s-wave', name='6_plots//sv2h_l_1200', L=L1200, s=5, delta=0.001, x0=0, _mu=(0, 0.01), _B=(0, 1.5), 
            t=time_1200, P=points_1200, N=5000, a=A_1200, v_map=Vmap_1200, Nv=nv_1200, filename=f_1200, xs_mu=(3, 6), xs_B=-0.8)

In [ ]:
p0 = dict(E=0, mu0=0.003, mu1=0.006, B=-0.8, delta=0.001, x0=0, N=10)

In [ ]:
Gana_1200_0 = analytical_xs_G(_sys=h_1200, _a =a/5, _W=W1000, _WSC=W1000, _L=L1200, _t=t, _alpha=np.sqrt(0.5), _beta=np.sqrt(0.5), p=p0)
Gnum_1200_0 = numerical_xs_G(_sys=h_1200, p1=p0)

In [ ]:
plot_xs_G(Ga=Gana_1200_0, Gn=Gnum_1200_0, name='d-wave-v2', L=L1200, s=4, ylim=1, p=p0)

# Cross sections s-wave 
# For $\nu = 4$ $(B = -0.8 T)$ $(\mu \approx [6; 9] meV)$

In [ ]:
plot_G_mu_B(E=0.0, title='s-wave', name='6_plots//sv4h_l_1200', L=L1200, s=5, delta=0.001, x0=0, _mu=(0, 0.01), _B=(0, 1.5), 
            t=time_1200, P=points_1200, N=5000, a=A_1200, v_map=Vmap_1200, Nv=nv_1200, filename=f_1200, xs_mu=(6, 9), xs_B=-0.8)

In [ ]:
p1 = dict(E=0, mu0=0.006, mu1=0.009, B=-0.8, delta=0.001, x0=0, N=500)

In [ ]:
Gana_1200_1 = analytical_xs_G(_sys=h_1200, _a =a, _W=W1000, _WSC=W1000, _L=L1200, _t=t, _alpha=0.5, _beta=0.5, p=p1)
Gnum_1200_1 = numerical_xs_G(_sys=h_1200, p1=p1)

In [ ]:
plot_xs_G(Ga=Gana_1200_1, Gn=Gnum_1200_1, name='d-wave-v4', L=L1200, s=4, ylim=2, p=p1)

# Cross sections s-wave 
# Further $(B = -0.8 T)$ $(\mu \approx [6; 20] meV)$

In [ ]:
p2 = dict(E=0, mu0=0.006, mu1=0.02, B=-0.8, delta=0.001, x0=0, N=500)

In [ ]:
Gana_1200_2 = analytical_xs_G(_sys=h_1200, _a =a, _W=W1000, _WSC=W1000, _L=L1200, _t=t, _alpha=0.5, _beta=0.5, p=p2)
Gnum_1200_2 = numerical_xs_G(_sys=h_1200, p1=p2)

In [ ]:
plot_xs_G(Ga=Gana_1200_2, Gn=Gnum_1200_2, name='d-wave-f', L=L1200, s=4, ylim=2, p=p2)